# Historical Features (`pipelines/03_historical_features.ipynb`)

This notebook reads accepted Silver transactions from `workspace.sentinel_silver.transactions` and computes point-in-time historical velocity and spending features per transaction (`trans_num`).

### Feature Definitions
| Feature | Entity Key | Window (`[lower, upper)`) | Semantics |
| :--- | :--- | :--- | :--- |
| `card_txn_count_1h` | `cc_num` | `[event_time - 1 hour, event_time)` | Earlier payment count for the same card in the previous hour (`0` when no history) |
| `card_txn_count_14d` | `cc_num` | `[event_time - 14 days, event_time)` | Earlier payment count for the same card over the previous 14 days (`0` when no history) |
| `card_avg_amount_14d` | `cc_num` | `[event_time - 14 days, event_time)` | Earlier average `amount` (`DECIMAL(18,2)`) for the same card over 14 days (`NULL` when no history) |
| `card_merchant_txn_count_1h` | `cc_num`, `merchant` | `[event_time - 1 hour, event_time)` | Earlier payment count for the same card and merchant in the previous hour (`0` when no history) |

### Scope & Guardrails
- **Window boundaries**: Includes the lower boundary (`event_time - window`); strictly excludes the current transaction timestamp (`event_time`) and all future events (`> event_time`).
- **Output grain**: Exactly one row per input `trans_num`.
- **Display only**: Does not write feature or Gold tables, modify Bronze/Silver tables, or release replay batches.

In [ ]:
# Step 1: Read accepted Silver transactions and verify input contract
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_TRANSACTIONS_TABLE = "workspace.sentinel_silver.transactions"
REQUIRED_SILVER_COLUMNS = {
    "trans_num",
    "cc_num",
    "merchant",
    "category",
    "amount",
    "event_time",
    "replay_batch",
}

silver_transactions = spark.read.table(SILVER_TRANSACTIONS_TABLE)

missing_cols = sorted(REQUIRED_SILVER_COLUMNS - set(silver_transactions.columns))
if missing_cols:
    raise ValueError(f"Missing required Silver columns: {missing_cols}")

if "is_fraud" in silver_transactions.columns:
    raise AssertionError("Label leakage guard failed: 'is_fraud' must not exist in Silver transactions.")

silver_row_count = silver_transactions.count()
silver_distinct_ids = silver_transactions.select("trans_num").distinct().count()
if silver_row_count != silver_distinct_ids:
    raise AssertionError(
        f"Input Silver grain violation: row_count ({silver_row_count}) != distinct trans_num ({silver_distinct_ids})"
    )

print(f"Loaded {SILVER_TRANSACTIONS_TABLE}: {silver_row_count:,} unique transactions.")

### Step 2: Point-in-Time Historical Feature Calculation

We convert `event_time` (`TIMESTAMP_NTZ`) to epoch seconds (`_event_seconds`) and apply range windows over `[-3600, -1]` seconds (`1 hour`) and `[-1209600, -1]` seconds (`14 days`):
- Because `rangeBetween(start, end)` is inclusive of both `current + start` and `current + end`, a range of `[-W, -1]` on second-granularity timestamps includes `event_time - W` and strictly excludes `event_time` (`offset 0`) as well as all future timestamps (`offset > 0`).
- Equal-timestamp transactions share the same `_event_seconds` value (`offset 0`) and are therefore excluded from each other's historical window.
- When no prior transactions exist in the 14-day window, `F.count("*")` evaluates to `0` and `F.avg("amount")` evaluates to `NULL`.

In [ ]:
# Step 2: Reusable historical feature transformation and grain validation
ONE_HOUR_SECONDS = 3600
FOURTEEN_DAYS_SECONDS = 14 * 86400  # 1,209,600 seconds

FEATURE_COLUMNS = [
    "card_txn_count_1h",
    "card_txn_count_14d",
    "card_avg_amount_14d",
    "card_merchant_txn_count_1h",
]


def compute_historical_features(df: DataFrame) -> DataFrame:
    """
    Compute point-in-time historical features per transaction.
    Windows include the lower boundary (-1h or -14d) and strictly exclude
    the current timestamp and all future events.
    """
    with_epoch = df.withColumn(
        "_event_seconds",
        F.expr("TIMESTAMPDIFF(SECOND, TIMESTAMP_NTZ '1970-01-01 00:00:00', event_time)"),
    )

    card_1h_window = (
        Window.partitionBy("cc_num")
        .orderBy("_event_seconds")
        .rangeBetween(-ONE_HOUR_SECONDS, -1)
    )
    card_14d_window = (
        Window.partitionBy("cc_num")
        .orderBy("_event_seconds")
        .rangeBetween(-FOURTEEN_DAYS_SECONDS, -1)
    )
    card_merchant_1h_window = (
        Window.partitionBy("cc_num", "merchant")
        .orderBy("_event_seconds")
        .rangeBetween(-ONE_HOUR_SECONDS, -1)
    )

    avg_14d_raw = F.avg("amount").over(card_14d_window)

    return (
        with_epoch.withColumn(
            "card_txn_count_1h",
            F.count(F.lit(1)).over(card_1h_window).cast("long"),
        )
        .withColumn(
            "card_txn_count_14d",
            F.count(F.lit(1)).over(card_14d_window).cast("long"),
        )
        .withColumn(
            "card_avg_amount_14d",
            F.when(F.col("card_txn_count_14d") > 0, F.round(avg_14d_raw, 2).cast("decimal(18,2)"))
            .otherwise(F.lit(None).cast("decimal(18,2)")),
        )
        .withColumn(
            "card_merchant_txn_count_1h",
            F.count(F.lit(1)).over(card_merchant_1h_window).cast("long"),
        )
        .drop("_event_seconds")
    )


def assert_feature_grain_and_null_semantics(
    input_df: DataFrame, feature_df: DataFrame
) -> dict[str, int]:
    """Verify 1:1 transaction grain and NULL/count consistency on computed features."""
    metrics = (
        feature_df.agg(
            F.count("*").alias("output_rows"),
            F.countDistinct("trans_num").alias("distinct_trans_num"),
            F.sum(F.when(F.col("card_txn_count_14d") == 0, 1).otherwise(0)).alias("no_14d_history_rows"),
            F.sum(F.when(F.col("card_txn_count_14d") > 0, 1).otherwise(0)).alias("with_14d_history_rows"),
            F.sum(
                F.when(
                    (F.col("card_txn_count_14d") == 0) & F.col("card_avg_amount_14d").isNotNull(),
                    1,
                ).otherwise(0)
            ).alias("invalid_non_null_avg_when_no_history"),
            F.sum(
                F.when(
                    (F.col("card_txn_count_14d") > 0) & F.col("card_avg_amount_14d").isNull(),
                    1,
                ).otherwise(0)
            ).alias("invalid_null_avg_when_history_exists"),
        )
        .collect()[0]
        .asDict()
    )

    input_rows = input_df.count()
    if metrics["output_rows"] != input_rows:
        raise AssertionError(f"Row count mismatch: input={input_rows}, output={metrics['output_rows']}")
    if metrics["distinct_trans_num"] != input_rows:
        raise AssertionError(
            f"Duplicate trans_num detected: output={metrics['output_rows']}, distinct={metrics['distinct_trans_num']}"
        )
    if metrics["invalid_non_null_avg_when_no_history"] != 0:
        raise AssertionError("Found non-NULL card_avg_amount_14d when card_txn_count_14d == 0")
    if metrics["invalid_null_avg_when_history_exists"] != 0:
        raise AssertionError("Found NULL card_avg_amount_14d when card_txn_count_14d > 0")

    return {k: int(v or 0) for k, v in metrics.items()}


silver_with_features = compute_historical_features(silver_transactions)
feature_metrics = assert_feature_grain_and_null_semantics(silver_transactions, silver_with_features)

print("Historical feature grain & NULL semantics verified:")
for k, v in feature_metrics.items():
    print(f"  {k}: {v:,}")

display(
    silver_with_features.select(
        "trans_num",
        "cc_num",
        "merchant",
        "amount",
        "event_time",
        *FEATURE_COLUMNS,
    ).orderBy("event_time", "trans_num").limit(25)
)

### Step 3: Synthetic Verification Suite

The synthetic suite below exercises `compute_historical_features` directly on an in-memory fixture to verify:
1. **No history**: First transaction for a card produces `0` counts and `NULL` `card_avg_amount_14d`.
2. **14-day lower boundary**: An event at exactly `event_time - 14 days` (`-1,209,600s`) is **included**; an event `1 second` earlier (`-1,209,601s`) is **excluded**.
3. **1-hour lower boundary & merchant specificity**: An event at `event_time - 1 hour` (`-3,600s`) is **included** in `card_txn_count_1h` (and `card_merchant_txn_count_1h` when the merchant matches); an event `1 second` earlier (`-3,601s`) is **excluded** from the 1-hour counts while remaining in the 14-day window.
4. **Equal timestamps excluded**: Two transactions at the exact same `event_time` do not count each other.
5. **Future-event exclusion**: Appending later transactions for the same card and merchant leaves all earlier feature rows 100% unchanged.

In [ ]:
# Step 3: Execute synthetic tests for boundaries, equal timestamps, no history, and future exclusion
from decimal import Decimal
from pyspark.sql import types as T

synthetic_schema = T.StructType([
    T.StructField("trans_num", T.StringType(), False),
    T.StructField("cc_num", T.StringType(), False),
    T.StructField("merchant", T.StringType(), False),
    T.StructField("category", T.StringType(), False),
    T.StructField("amount", T.DecimalType(18, 2), False),
    T.StructField("event_time", T.TimestampNTZType(), False),
])

# Anchor time T0 = 2019-01-15 12:00:00
# - T_before_14d: 2019-01-01 11:59:59 (-14d - 1s from T0 -> excluded at T0)
# - T_exact_14d:  2019-01-01 12:00:00 (exact -14d from T0 -> included at T0)
# - T_before_1h:  2019-01-15 10:59:59 (-1h - 1s from T0 -> excluded from 1h at T0, included in 14d)
# - T_exact_1h:   2019-01-15 11:00:00 (exact -1h from T0, same merchant M1 -> included in 1h & merchant 1h)
# - T_other_merch:2019-01-15 11:30:00 (-30m from T0, different merchant M2 -> included in card 1h, excluded from M1 1h)
# - T0_a & T0_b:  2019-01-15 12:00:00 (equal timestamps -> must not count each other)
base_synthetic_rows = [
    ("tx_1_before_14d", "card_A", "M1", "grocery", Decimal("10.00"), "2019-01-01 11:59:59"),
    ("tx_2_exact_14d",  "card_A", "M1", "grocery", Decimal("20.00"), "2019-01-01 12:00:00"),
    ("tx_3_before_1h",  "card_A", "M1", "grocery", Decimal("40.00"), "2019-01-15 10:59:59"),
    ("tx_4_exact_1h",   "card_A", "M1", "grocery", Decimal("60.00"), "2019-01-15 11:00:00"),
    ("tx_5_other_merch","card_A", "M2", "gas",     Decimal("80.00"), "2019-01-15 11:30:00"),
    ("tx_6_t0_a",       "card_A", "M1", "grocery", Decimal("100.00"), "2019-01-15 12:00:00"),
    ("tx_7_t0_b",       "card_A", "M1", "grocery", Decimal("200.00"), "2019-01-15 12:00:00"),
]

synthetic_base_df = (
    spark.createDataFrame(
        [(t, c, m, cat, amt, ts) for t, c, m, cat, amt, ts in base_synthetic_rows],
        schema=[
            "trans_num", "cc_num", "merchant", "category", "amount", "event_time_str"
        ],
    )
    .withColumn("event_time", F.to_timestamp_ntz("event_time_str"))
    .drop("event_time_str")
)

synthetic_features_df = compute_historical_features(synthetic_base_df)
assert_feature_grain_and_null_semantics(synthetic_base_df, synthetic_features_df)

actual_by_tx = {
    row["trans_num"]: (
        row["card_txn_count_1h"],
        row["card_txn_count_14d"],
        row["card_avg_amount_14d"],
        row["card_merchant_txn_count_1h"],
    )
    for row in synthetic_features_df.collect()
}

expected_by_tx = {
    # No history: 0 counts and NULL 14d average
    "tx_1_before_14d": (0, 0, None, 0),
    # 1 second later: sees tx_1 in 1h, 14d, and M1 1h
    "tx_2_exact_14d":  (1, 1, Decimal("10.00"), 1),
    # At 2019-01-15 10:59:59: both tx_1 and tx_2 are within 14d (avg=(10+20)/2=15.00), none within 1h
    "tx_3_before_1h":  (0, 2, Decimal("15.00"), 0),
    # At 2019-01-15 11:00:00: tx_3 is 1s ago (in 1h & M1 1h); tx_1, tx_2, tx_3 all in 14d (avg=(10+20+40)/3=23.33)
    "tx_4_exact_1h":   (1, 3, Decimal("23.33"), 1),
    # At 2019-01-15 11:30:00 (merchant M2): tx_3 & tx_4 in card 1h (2), 0 in M2 1h; tx_1..tx_4 in 14d (avg=130/4=32.50)
    "tx_5_other_merch":(2, 4, Decimal("32.50"), 0),
    # At T0 (2019-01-15 12:00:00):
    # - tx_1 (-14d - 1s) is EXCLUDED from 14d
    # - tx_2 (exact -14d), tx_3, tx_4, tx_5 are INCLUDED in 14d -> 4 txns, avg=(20+40+60+80)/4 = 50.00
    # - tx_3 (-1h - 1s) is EXCLUDED from 1h
    # - tx_4 (exact -1h, M1) and tx_5 (-30m, M2) are INCLUDED in card 1h -> 2 txns
    # - Only tx_4 is in (card_A, M1) 1h -> 1 txn
    # - Equal-timestamp peers tx_6_t0_a and tx_7_t0_b EXCLUDE each other
    "tx_6_t0_a":       (2, 4, Decimal("50.00"), 1),
    "tx_7_t0_b":       (2, 4, Decimal("50.00"), 1),
}

for tx_id, expected_tuple in expected_by_tx.items():
    actual_tuple = actual_by_tx.get(tx_id)
    if actual_tuple != expected_tuple:
        raise AssertionError(
            f"Synthetic check failed for {tx_id}: expected {expected_tuple}, got {actual_tuple}"
        )

# Future-event exclusion test: append future events and verify earlier rows do not change
future_rows = [
    ("tx_8_future_1m", "card_A", "M1", "grocery", Decimal("999.00"), "2019-01-15 12:01:00"),
    ("tx_9_future_1d", "card_A", "M1", "grocery", Decimal("500.00"), "2019-01-16 12:00:00"),
]
synthetic_with_future_df = synthetic_base_df.unionByName(
    spark.createDataFrame(
        future_rows,
        schema=["trans_num", "cc_num", "merchant", "category", "amount", "event_time_str"],
    )
    .withColumn("event_time", F.to_timestamp_ntz("event_time_str"))
    .drop("event_time_str")
)

features_after_future_added = (
    compute_historical_features(synthetic_with_future_df)
    .join(synthetic_base_df.select("trans_num"), on="trans_num", how="inner")
    .select(synthetic_features_df.columns)
)

diff_forward = synthetic_features_df.exceptAll(features_after_future_added).count()
diff_reverse = features_after_future_added.exceptAll(synthetic_features_df).count()
if diff_forward != 0 or diff_reverse != 0:
    raise AssertionError(
        f"Future-event exclusion violated: forward_diff={diff_forward}, reverse_diff={diff_reverse}"
    )

print("All synthetic tests passed (boundaries, equal timestamps, no-history NULLs, future exclusion).")
display(synthetic_features_df.orderBy("event_time", "trans_num"))